# Implied Volatility Surface Builder
### SPY + QQQ + IWM Options | Gaussian Process + MLP | Arbitrage Checks

This notebook walks through the full pipeline step by step:
1. Fetch live options data
2. Extract implied volatility via Black-Scholes inversion
3. Fit GP and MLP surfaces
4. Check arbitrage conditions
5. Visualize results

In [1]:
import sys, os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath('.'))
np.random.seed(42)
print('Libraries loaded ✓')

Libraries loaded ✓


## Step 1 — Fetch Options Data

We pull live delayed options chains from SPY, QQQ, and IWM via yfinance.
All strikes are converted to **moneyness space** (K/S) so data from different
underlyings can be combined meaningfully.

In [2]:
from data.fetch_options import fetch_spy_options

S, r, df_raw = fetch_spy_options(["SPY", "QQQ", "IWM"])
print(f"\nSpot (SPY): ${S:.2f}")
print(f"Risk-free rate: {r*100:.2f}%")
print(f"Total contracts: {len(df_raw)}")
df_raw.head(10)

[Data] Risk-free rate: 3.59%
[Data] SPY spot: $710.30
[Data] SPY raw contracts: 8162
[Data] QQQ spot: $659.30
[Data] QQQ raw contracts: 6953
[Data] IWM spot: $272.33
[Data] IWM raw contracts: 3418

[Data] Total contracts: 12636
underlying
IWM    2375
QQQ    4803
SPY    5458
[Data] Expirations: 30

Spot (SPY): $710.30
Risk-free rate: 3.59%
Total contracts: 12636


,strike,expiry,tte,mid_price,option_type,moneyness,underlying,spot,yf_iv
0,550.0,2026-04-30,0.0,161.820,call,0.774321,SPY,710.299988,1.988281
1,555.0,2026-04-30,0.0,156.830,call,0.781360,SPY,710.299988,1.931641
2,560.0,2026-04-30,0.0,151.855,call,0.788399,SPY,710.299988,1.878907
3,565.0,2026-04-30,0.0,146.830,call,0.795439,SPY,710.299988,1.814454
4,570.0,2026-04-30,0.0,141.860,call,0.802478,SPY,710.299988,1.763185
5,575.0,2026-04-30,0.0,136.815,call,0.809517,SPY,710.299988,1.695802
6,580.0,2026-04-30,0.0,131.830,call,0.816556,SPY,710.299988,1.641603
7,585.0,2026-04-30,0.0,126.860,call,0.823596,SPY,710.299988,1.590334
8,590.0,2026-04-30,0.0,121.860,call,0.830635,SPY,710.299988,1.533205
9,591.0,2026-04-30,0.0,120.815,call,0.832043,SPY,710.299988,1.512698


In [3]:
# Distribution of contracts by underlying and expiry
import plotly.express as px

fig = px.histogram(df_raw, x='tte', color='underlying', nbins=30,
                   title='Contract Distribution by Time to Expiry',
                   labels={'tte': 'Time to Expiry (years)'},
                   template='plotly_dark')
fig.show()

## Step 2 — Black-Scholes IV Extraction

For each contract we solve numerically for σ such that:

$$BS(\sigma) - C_{market} = 0$$

Using **Brent's method** — a bracketing root-finder guaranteed to converge.

The Black-Scholes call formula:
$$C = S \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$
$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [4]:
# Quick sanity check first
from iv.extractor import bs_call, extract_iv

true_sigma = 0.20
price = bs_call(S=100, K=100, T=0.25, r=0.05, sigma=true_sigma)
recovered = extract_iv(price, S=100, K=100, T=0.25, r=0.05, option_type='call')
print(f"True σ:      {true_sigma:.6f}")
print(f"Recovered σ: {recovered:.6f}")
print(f"Error:       {abs(recovered - true_sigma):.2e}")

True σ:      0.200000
Recovered σ: 0.200000
Error:       8.01e-09


In [5]:
from iv.extractor import extract_iv_surface

df = extract_iv_surface(df_raw, S, r)
print(f"\nClean IV dataset: {len(df)} contracts")
print(f"IV range: {df['iv'].min():.1%} – {df['iv'].max():.1%}")
print(f"Median IV: {df['iv'].median():.1%}")

[IV] Extracting IV for 12636 contracts...


Brent root-finding: 100%|██████████| 12636/12636 [00:43<00:00, 293.46it/s]

[IV] Success: 7380/7380 | Range: 5.0%–55.4%

Clean IV dataset: 7380 contracts
IV range: 5.0% – 55.4%
Median IV: 24.0%


In [7]:
# IV distribution by underlying
fig = px.box(df, x='underlying', y='iv', color='underlying',
             title='IV Distribution by Underlying',
             labels={'iv': 'Implied Volatility'},
             template='plotly_dark')
fig.update_yaxes(tickformat='.0%')
fig.show()

In [9]:
# IV smile — raw market data scatter
fig = px.scatter(df, x='moneyness', y='iv', color='underlying',
                 facet_col='option_type', opacity=0.6,
                 title='Raw IV Smile — Market Data',
                 labels={'iv': 'Implied Volatility', 'moneyness': 'Moneyness (K/S)'},
                 template='plotly_dark')
fig.update_yaxes(tickformat='.0%')
fig.add_vline(x=1.0, line_dash='dot', line_color='orange', annotation_text='ATM')
fig.show()

## Step 3a — Gaussian Process Surface

GP models IV as a random function with prior:
$$\sigma(K,T) \sim \mathcal{GP}(0, k(x, x'))$$

Kernel: **Anisotropic RBF** — separate length scales for moneyness and TTE,
plus a WhiteKernel for observation noise.

Key advantage: **probabilistic output** — gives mean + uncertainty band.

In [10]:
from models.gp_surface import IVSurfaceGP

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

gp = IVSurfaceGP(n_restarts=5)
gp.fit(df_train)

gp_metrics = gp.evaluate(df_test)
print(f"\nGP Test Results:")
print(f"  MAE:  {gp_metrics['MAE']:.4f} ({gp_metrics['MAE']*100:.2f}% IV)")
print(f"  RMSE: {gp_metrics['RMSE']:.4f}")
print(f"  R²:   {gp_metrics['R2']:.4f}")

[GP] Subsampled to 500 points
[GP] Fitting on 500 points...
[GP] Train R²: 0.9261

GP Test Results:
  MAE:  0.0321 (3.21% IV)
  RMSE: 0.0583
  R²:   0.7216


## Step 3b — MLP Surface

Architecture: `Input(3) → 64 → 128 → 64 → Softplus(1)`

- **Softplus** output guarantees IV > 0 smoothly
- **BatchNorm** + **Dropout** for regularization  
- **Cosine annealing** LR schedule
- **Early stopping** on validation loss

In [11]:
from models.mlp_surface import IVSurfaceMLP

mlp = IVSurfaceMLP(epochs=300, patience=30)
mlp.fit(df_train)

mlp_metrics = mlp.evaluate(df_test)
print(f"\nMLP Test Results:")
print(f"  MAE:  {mlp_metrics['MAE']:.4f} ({mlp_metrics['MAE']*100:.2f}% IV)")
print(f"  RMSE: {mlp_metrics['RMSE']:.4f}")
print(f"  R²:   {mlp_metrics['R2']:.4f}")

[MLP] Training on cpu | 5018 train, 886 val
[MLP] Epoch   1 | train: 0.749631 | val: 0.576575
[MLP] Epoch  50 | train: 0.504102 | val: 0.489693
[MLP] Epoch 100 | train: 0.500017 | val: 0.484038
[MLP] Early stop at epoch 117 | best val: 0.479638

MLP Test Results:
  MAE:  0.0610 (6.10% IV)
  RMSE: 0.0796
  R²:   0.4812


In [12]:
import importlib
import models.mlp_surface as mlp_mod
importlib.reload(mlp_mod)
from models.mlp_surface import IVSurfaceMLP

mlp = IVSurfaceMLP(epochs=300, patience=30)
mlp.fit(df_train)

[MLP] Training on cpu | 5018 train, 886 val
[MLP] Epoch   1 | train: 0.764157 | val: 0.576357
[MLP] Epoch  50 | train: 0.504503 | val: 0.489389
[MLP] Epoch 100 | train: 0.492989 | val: 0.484122
[MLP] Early stop at epoch 126 | best val: 0.479426


In [13]:
# Training loss curve
fig = go.Figure()
fig.add_trace(go.Scatter(y=mlp.history['train_loss'], name='Train Loss',
                         line=dict(color='#636EFA')))
fig.add_trace(go.Scatter(y=mlp.history['val_loss'], name='Val Loss',
                         line=dict(color='#EF553B')))
fig.update_layout(title='MLP Training History', xaxis_title='Epoch',
                  yaxis_title='MSE Loss', template='plotly_dark')
fig.show()

## Step 4 — Build and Visualize IV Surfaces

In [14]:
m_grid = np.linspace(0.80, 1.20, 60)
t_grid = np.linspace(7/365, 1.0, 40)

iv_gp, iv_gp_std = gp.predict_grid(m_grid, t_grid)
iv_mlp           = mlp.predict_grid(m_grid, t_grid)

print(f"GP surface shape: {iv_gp.shape}")
print(f"IV range (GP):  {iv_gp.min():.1%} – {iv_gp.max():.1%}")
print(f"IV range (MLP): {iv_mlp.min():.1%} – {iv_mlp.max():.1%}")

GP surface shape: (60, 40)
IV range (GP):  10.9% – 54.9%
IV range (MLP): 26.3% – 46.0%


In [15]:
# 3D Surface — GP
from plotly.subplots import make_subplots

M, T = np.meshgrid(m_grid, t_grid, indexing='ij')
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'surface'},{'type':'surface'}]],
                    subplot_titles=['Gaussian Process', 'MLP'])
fig.add_trace(go.Surface(x=M, y=T*365, z=iv_gp*100, colorscale='Viridis',
                         colorbar=dict(x=0.46, title='IV %')), row=1, col=1)
fig.add_trace(go.Surface(x=M, y=T*365, z=iv_mlp*100, colorscale='Plasma',
                         colorbar=dict(x=1.02, title='IV %')), row=1, col=2)
fig.update_layout(title='SPY IV Surface — GP vs MLP', height=600, template='plotly_dark',
                  scene=dict(xaxis_title='Moneyness', yaxis_title='Days', zaxis_title='IV %'),
                  scene2=dict(xaxis_title='Moneyness', yaxis_title='Days', zaxis_title='IV %'))
fig.show()

In [16]:
# GP Uncertainty Band at fixed maturity
idx = 10
T_val = t_grid[idx]
iv_slice  = iv_gp[:, idx] * 100
std_slice = iv_gp_std[:, idx] * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=m_grid, y=iv_slice+std_slice, line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=m_grid, y=iv_slice-std_slice, line=dict(width=0),
                         fill='tonexty', fillcolor='rgba(99,110,250,0.2)', name='±1σ'))
fig.add_trace(go.Scatter(x=m_grid, y=iv_slice, line=dict(color='#636EFA', width=2.5), name='GP mean'))
fig.add_vline(x=1.0, line_dash='dot', line_color='orange', annotation_text='ATM')
fig.update_layout(title=f'GP Uncertainty Band — T={T_val*365:.0f} days',
                  xaxis_title='Moneyness (K/S)', yaxis_title='IV (%)',
                  template='plotly_dark', height=450)
fig.show()

## Step 5 — Arbitrage Checks

**Calendar spread:** Total variance $w(K,T) = \sigma^2 \cdot T$ must be non-decreasing in $T$.

**Butterfly spread:** $w(x,T)$ must be convex in log-moneyness $x = \log(K/S)$.

In [17]:
from arbitrage.checks import run_arbitrage_checks

print("GP Surface:")
gp_arb = run_arbitrage_checks(iv_gp, m_grid, t_grid)

print("\nMLP Surface:")
mlp_arb = run_arbitrage_checks(iv_mlp, m_grid, t_grid)

GP Surface:
[Arbitrage] Calendar check...
[Arbitrage] Butterfly check...
═══════════════════════════════════
   ARBITRAGE CHECK REPORT
═══════════════════════════════════
Calendar  violations: 514  (21.4%)
Butterfly violations: 941 (39.2%)
⚠ Violations detected
═══════════════════════════════════

MLP Surface:
[Arbitrage] Calendar check...
[Arbitrage] Butterfly check...
═══════════════════════════════════
   ARBITRAGE CHECK REPORT
═══════════════════════════════════
Calendar  violations: 0  (0.0%)
Butterfly violations: 58 (2.4%)
⚠ Violations detected
═══════════════════════════════════


## Final Results

In [18]:
results = pd.DataFrame({
    'Model': ['Gaussian Process', 'MLP'],
    'MAE':   [f"{gp_metrics['MAE']:.4f}", f"{mlp_metrics['MAE']:.4f}"],
    'RMSE':  [f"{gp_metrics['RMSE']:.4f}", f"{mlp_metrics['RMSE']:.4f}"],
    'R²':    [f"{gp_metrics['R2']:.4f}", f"{mlp_metrics['R2']:.4f}"],
    'Calendar Violations':  [gp_arb.calendar_violations,  mlp_arb.calendar_violations],
    'Butterfly Violations': [gp_arb.butterfly_violations, mlp_arb.butterfly_violations],
})
results

,Model,MAE,RMSE,R²,Calendar Violations,Butterfly Violations
0,Gaussian Process,0.0321,0.0583,0.7216,514,941
1,MLP,0.0610,0.0796,0.4812,0,58
